# Check the documented examples

Run this notebook from an environment with the repository installed, for example with `python -m pip install -e ".[tests]"`. The prediction and fitting examples use synthetic data; the station example uses the small fixture under `tests/tide_stations`. The gridded-model example needs a separately installed local model database. Set `CHT_TIDE_MODEL_DATABASE` to its path to run that example; otherwise the notebook reports it as skipped.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

repository_root = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "cht_tide").is_dir() and (parent / "tests" / "tide_stations").is_dir()
)
print(f"Repository: {repository_root}")

## Predict from constituent amplitudes and phases

This is the README and getting-started prediction example.

In [ ]:
from cht_tide import predict

constituents = pd.DataFrame(
    {
        "amplitude": [0.85, 0.22, 0.18, 0.15, 0.12],
        "phase": [120.0, 145.0, 100.0, 30.0, 350.0],
    },
    index=["M2", "S2", "N2", "K1", "O1"],
)
quick_times = pd.date_range("2024-01-01", periods=24, freq="h")
quick_water_levels = predict(constituents, quick_times)
assert len(quick_water_levels) == len(quick_times)

# Also run the getting-started example's seven-day, ten-minute time axis.
times = pd.date_range("2024-01-01", periods=7 * 144, freq="10min")
water_levels = predict(constituents, times)

assert len(water_levels) == len(times)
assert np.isfinite(water_levels).all()
print(f"Predicted {len(water_levels)} finite water levels")

### Plot the prediction

Plot the first two days of the predicted series so the tidal variation is easy to inspect.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times[:289], water_levels[:289])
ax.set(title="Predicted water level", xlabel="Time", ylabel="Water level (m)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

## Fit constituents to observations

Create a deterministic synthetic observation series, then run the documented `Tide.decompose` call and inspect its structured result.

In [ ]:
from cht_tide.tide import Tide

observation_times = pd.date_range("2024-01-01", periods=30 * 24 + 1, freq="h")
known_constituents = constituents.loc[["M2", "S2"]]
observations = predict(known_constituents, observation_times)
fit = Tide.decompose(heights=observations, t=observation_times.to_pydatetime())

assert len(fit.model) > 1
assert {"constituent", "amplitude", "phase"}.issubset(fit.model.dtype.names)
print(f"Fitted {len(fit.model) - 1} tidal constituents")
fit.model[:3]

### View the fitted values as a table

The table includes the mean-level (`Z0`) row followed by the fitted tidal constituents.

In [ ]:
fit_table = pd.DataFrame(
    {
        "amplitude (m)": fit.model["amplitude"],
        "phase (degrees)": fit.model["phase"],
    },
    index=[item.name for item in fit.model["constituent"]],
)
fit_table.index.name = "constituent"

fit_table

## Read station data

This exercises the documented station database and component lookup against the repository's small test dataset.

In [ ]:
from cht_tide.tide_stations import TideStationsDatabase

db = TideStationsDatabase(path=repository_root / "tests" / "tide_stations")
dataset = db.dataset["xtide_free"]
stations = dataset.get_gdf()
station_name = stations.iloc[0]["name"]
station_constituents = dataset.get_components(name=station_name)

assert not stations.empty
assert not station_constituents.empty
assert {"amplitude", "phase"}.issubset(station_constituents.columns)
print(f"Read {len(station_constituents)} constituents for {station_name}")

## Interpolate a gridded tide model (optional)

The package does not include gridded model files. Configure `CHT_TIDE_MODEL_DATABASE` to run this cell against an available local database.

In [ ]:
model_database_path = r"c:\DelftDashboard Training\data\tide_models"

s3_bucket = "deltares-ddb"
s3_key = "data/tide_models"

if not model_database_path:
    print("SKIPPED: set CHT_TIDE_MODEL_DATABASE to test gridded model interpolation.")
else:
    from cht_tide.database import TideModelDatabase

    db = TideModelDatabase(
        path=model_database_path,
        # s3_bucket=s3_bucket,
        # s3_key=s3_key,
        # check_online=True,
    )

    # db = TideModelDatabase(path=model_database_path)
    model = db.get_dataset("fes2014_extrap")
    assert model is not None, "No fes2014 dataset found in the configured database"
    # assert all(
    #     (Path(model.path) / filename).is_file() for filename in model.files
    # ), "Model files must already exist locally; this notebook does not download data"
    point_data = model.get_data_on_points(
        x=[4.0, 5.0],
        y=[52.0, 53.0],
        crs="EPSG:4326",
        format="dataframe",
        constituents="main",
    )
    assert len(point_data) == 2
    assert all(not point.empty for point in point_data)
    print(f"Interpolated constituents at {len(point_data)} points")